<a href="https://colab.research.google.com/github/DiyaMalik07/SC_Project/blob/main/AGA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q pymoo

import numpy as np
import pandas as pd
import time
import urllib.request
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score

from pymoo.core.problem import Problem
from pymoo.core.sampling import Sampling
from pymoo.core.callback import Callback
from pymoo.algorithms.soo.nonconvex.ga import GA
from pymoo.operators.crossover.hux import HalfUniformCrossover
from pymoo.operators.mutation.bitflip import BitflipMutation
from pymoo.optimize import minimize

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print("Libraries imported successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.2/347.2 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 879.5/879.5 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 5.7 MB/s eta 0:00:00
Libraries imported successfully!


In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/internet_ads/ad.data"
urllib.request.urlretrieve(url, "ad.data")

df = pd.read_csv("ad.data", header=None, low_memory=False)
print(f"Dataset shape (rows, columns): {df.shape}")

print("Preprocessing data")
df = df.replace(r'^\s*\?\s*$', np.nan, regex=True)

label_col = df.columns[-1]
y = df[label_col].astype(str).str.strip().apply(
    lambda v: 1 if v.lower().startswith('ad') else 0
).values

X = df.drop(columns=[label_col])
X = X.apply(pd.to_numeric, errors='coerce')
X = X.fillna(X.mean())
X = X.values

n_features = X.shape[1]
print(f"Total features: {n_features}")
print(f"Total instances: {X.shape[0]}")
print(f"Class balance -> ad: {sum(y==1)}  |  nonad: {sum(y==0)}")

Dataset shape (rows, columns): (3279, 1559)
Preprocessing data
Total features: 1558
Total instances: 3279
Class balance -> ad: 459  |  nonad: 2820


In [ ]:
print("Splitting data (Train 60% / Validation 20% / Test 20%)...")

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=RANDOM_SEED, stratify=y_temp
)

print(f"Train samples: {X_train.shape[0]}")
print(f"Validation samples: {X_val.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")

Splitting data (Train 60% / Validation 20% / Test 20%)...
Train samples: 1967
Validation samples: 656
Test samples: 656


In [ ]:
print("Training Baseline Random Forest on ALL features")

start_time = time.time()
baseline_model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1)
baseline_model.fit(X_train, y_train)
baseline_train_time = time.time() - start_time

baseline_predictions = baseline_model.predict(X_test)
baseline_accuracy = accuracy_score(y_test, baseline_predictions)
baseline_f1 = f1_score(y_test, baseline_predictions)

print(f"Baseline Accuracy : {baseline_accuracy:.4f}")
print(f"Baseline F1-score : {baseline_f1:.4f}")
print(f"Features used     : {n_features}")
print(f"Training time     : {baseline_train_time:.2f} seconds")

Training Baseline Random Forest on ALL features
Baseline Accuracy : 0.9665
Baseline F1-score : 0.8690
Features used     : 1558
Training time     : 2.25 seconds


In [ ]:
print("Setting up Adaptive Genetic Algorithm (AGA)")
class DiverseBinarySampling(Sampling):
    def _do(self, problem, n_samples, **kwargs):
        population = np.zeros((n_samples, problem.n_var), dtype=bool)
        for i in range(n_samples):
            selection_prob = np.random.uniform(0.02, 0.80)
            population[i] = np.random.rand(problem.n_var) < selection_prob
            if population[i].sum() == 0:
                population[i, np.random.randint(0, problem.n_var)] = True
        return population

class AGACallback(Callback):
    def __init__(self, base_mut_prob, base_cx_prob):
        super().__init__()
        self.base_mut_prob = base_mut_prob
        self.base_cx_prob = base_cx_prob

    def notify(self, algorithm):
        F = algorithm.pop.get("F")
        if F is not None and len(F) > 0:
            f_best = np.min(F)
            f_avg = np.mean(F)

            # Calculate diversity (spread between average fitness and best fitness)
            spread = f_avg - f_best

            # ADAPTIVE LOGIC: If population is stuck (low spread), boost mutation to escape
            if spread < 0.01:
                new_mut_prob = min(0.5, self.base_mut_prob * 5.0)
                new_cx_prob = max(0.4, self.base_cx_prob * 0.8)
            else:
                new_mut_prob = self.base_mut_prob
                new_cx_prob = self.base_cx_prob

            algorithm.mating.mutation = BitflipMutation(prob=new_mut_prob)
            algorithm.mating.crossover = HalfUniformCrossover(prob=new_cx_prob)

class AGAFeatureSelectionProblem(Problem):
    def __init__(self, X_train, y_train, X_val, y_val, alpha=0.95):
        super().__init__(n_var=n_features, n_obj=1, n_constr=0, xl=0, xu=1, vtype=bool)
        self.X_train, self.y_train = X_train, y_train
        self.X_val, self.y_val = X_val, y_val
        self.alpha = alpha

    def _evaluate(self, population, out, *args, **kwargs):
        fitness_list = []
        for individual in population:
            mask = individual.astype(bool)
            if mask.sum() == 0:
                mask[np.random.randint(0, self.n_var)] = True

            model = RandomForestClassifier(n_estimators=30, random_state=RANDOM_SEED, n_jobs=-1)
            model.fit(self.X_train[:, mask], self.y_train)
            predictions = model.predict(self.X_val[:, mask])

            f1 = f1_score(self.y_val, predictions)
            fitness = (self.alpha * (1.0 - f1)) + ((1.0 - self.alpha) * (mask.sum() / self.n_var))
            fitness_list.append(fitness)

        out["F"] = np.column_stack([fitness_list])

Setting up Adaptive Genetic Algorithm (AGA)


In [ ]:
POPULATION_SIZE = 40
GENERATIONS = 30

base_mutation_rate = 5.0 / n_features
base_crossover_rate = 0.9

aga_problem = AGAFeatureSelectionProblem(X_train, y_train, X_val, y_val, alpha=0.95)

aga_algorithm = GA(
    pop_size=POPULATION_SIZE,
    sampling=DiverseBinarySampling(),
    crossover=HalfUniformCrossover(prob=base_crossover_rate),
    mutation=BitflipMutation(prob=base_mutation_rate),
    eliminate_duplicates=True
)

print(f"Running AGA optimization for {GENERATIONS} generations...")
start_time = time.time()

aga_result = minimize(
    aga_problem,
    aga_algorithm,
    termination=('n_gen', GENERATIONS),
    callback=AGACallback(base_mutation_rate, base_crossover_rate),
    seed=RANDOM_SEED,
    verbose=True
)

aga_run_time = time.time() - start_time
print(f"\nAGA finished in {aga_run_time:.2f} seconds")

Running AGA optimization for 30 generations...
n_gen  |  n_eval  |     f_avg     |     f_min    
     1 |       40 |  0.1582475666 |  0.0722278598
     2 |       80 |  0.0951430745 |  0.0722278598
     3 |      120 |  0.0848879336 |  0.0691477678
     4 |      160 |  0.0798731820 |  0.0654589398
     5 |      200 |  0.0753824471 |  0.0617657289
     6 |      240 |  0.0721291565 |  0.0616373592
     7 |      280 |  0.0691943055 |  0.0552590343
     8 |      320 |  0.0674683342 |  0.0519856068
     9 |      360 |  0.0650993282 |  0.0519856068
    10 |      400 |  0.0642941893 |  0.0519856068
    11 |      440 |  0.0623436389 |  0.0519856068
    12 |      480 |  0.0619115736 |  0.0519856068
    13 |      520 |  0.0615132043 |  0.0519856068
    14 |      560 |  0.0610611279 |  0.0519856068
    15 |      600 |  0.0606270492 |  0.0519856068
    16 |      640 |  0.0603840017 |  0.0519856068
    17 |      680 |  0.0598580247 |  0.0519856068
    18 |      720 |  0.0596975506 |  0.0519856068
   

In [ ]:
print("Evaluating best feature subset found by AGA.\n")
aga_best_mask = aga_result.X.astype(bool)
aga_features_count = int(aga_best_mask.sum())

start_time = time.time()
aga_final_model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1)
aga_final_model.fit(X_train[:, aga_best_mask], y_train)
aga_final_train_time = time.time() - start_time

aga_predictions = aga_final_model.predict(X_test[:, aga_best_mask])
aga_accuracy = accuracy_score(y_test, aga_predictions)
aga_f1 = f1_score(y_test, aga_predictions)

print("FINAL RESULTS: Random FOrest vs AGA")
print(f"{'Metric':<28}{'All Features':<18}{'AGA-Selected':<18}")
print(f"{'Accuracy':<28}{baseline_accuracy:<18.4f}{aga_accuracy:<18.4f}")
print(f"{'F1-score':<28}{baseline_f1:<18.4f}{aga_f1:<18.4f}")
print(f"{'Number of Features':<28}{n_features:<18}{aga_features_count:<18}")

reduction_pct = (1 - aga_features_count / n_features) * 100
print(f"{'Feature Reduction %':<28}{'-':<18}{reduction_pct:<18.1f}")
print(f"{'Search Algorithm Time (s)':<28}{'-':<18}{aga_run_time:<18.2f}")
print(f"{'Model Training Time (s)':<28}{baseline_train_time:<18.2f}{aga_final_train_time:<18.2f}")

Evaluating best feature subset found by AGA.

FINAL RESULTS: Random FOrest vs AGA
Metric                      All Features      AGA-Selected      
Accuracy                    0.9665            0.9680            
F1-score                    0.8690            0.8772            
Number of Features          1558              793               
Feature Reduction %         -                 49.1              
Search Algorithm Time (s)   -                 379.63            
Model Training Time (s)     2.25              1.13              
